In [1]:
import langchain
import langchain_core
import langchain_community
print(langchain.__version__)
print(langchain_core.__version__)
print(langchain_community.__version__)

1.2.3
1.2.7
0.4.1


In [27]:
"""
#Project structure
#packages to work - wth csv, pdf docs, langchain or other frameworks,streamlit etc.
#refer to requirements.txt file
#analysis using csv file > advanced_analysis
#building llM appl using fwk such as langchain

--llm defn (client = model defin)
--scenario = prompt templates + (other options:
     Prompt** (methods to interact with LLM: invoke/chat model)
    --structure
    --templates
    --specific role assignment
    --instructions
    --input examples /Using (RAG)
    --output format/template
    --techniques to be used
    --parameter settings/output generation settings/safety settings
    --request for reasoning/validation/citations in terms of explainations from LLM for the response it generated)
> prompt will use analysis + question to be sent > to LLM
--prompt = PromptTemplate(template = scenario,input_variables=['analysis','question'])
--question = 'Look at my analysis and give me a recommendation to improve my sales in every region'
--create a chain.. LLMChain(prompt=prompt,llm=llm defn) --older version & chain.run(analysis, question)
--create a chain.. as per new version
---example
# # RunnableSequence
chain = prompt | client

for q in questions:
     print(f"\nQ: {q}")
     print(chain.invoke({"question": q}))
---example ends----
-- run the chain as per new version --> chain.invoke({"question": q})


#Developing a chain which will run the tasks sequentially
#1
data_analysis_template = "analyze my data as in {analysis} & provide a initial summary"
prompt_a = PromptTemplate(template = data_analysis_template,input_variables=['analysis'])
create a chain.. 
chain1 = LLMChain(prompt=prompt,llm=llm defn,output = 'output')

#2 

recommendation_template = "based on my {analysis} data & provide a recommendation to addres questions"
prompt_a = PromptTemplate(template = recommendation_template,input_variables=['output','question'])
create a chain.. 
chain2 = LLMChain(prompt=prompt,llm=llm defn,output = 'output2')
chain.run(analysis, question)

overall_chain = SequentialChain(
                chains = [chain1,chain2],
                input_variables = [],
                output_variables = [])
question = "  "
run the overall_chain

#work with pdf documents > load > splitting >embedding > storing in RAG layer ie vector store 
#build a retriever which extracts info from rag
#work with pdf documents > load > splitting >embedding > storing in RAG layer ie vector store 
#build a retriever which extracts info from rag
#Implementation of RAG based layer
loaders
splitters > chunking
embedding
stored in vector store
building retriever
  code refer in git for example
     retriever = vectorstore.as_retriever(search_type="similarity", k=4)
     qa_chain = RetrievalQA.from_chain_type(
     llm=llm,
     chain_type='stuff',
     retriever = retriever,
     return_source_documents = True)

#implementation of a tool or call a tool like wikipedia..
#implementation of memory

#evaluation using qaevalchain

#visualization
"""



'\n#Project structure\n#packages to work - wth csv, pdf docs, langchain or other frameworks,streamlit etc.\n#refer to requirements.txt file\n#analysis using csv file > advanced_analysis\n#building llM appl using fwk such as langchain\n\n--llm defn (client = model defin)\n--scenario = prompt templates + (other options:\n     Prompt** (methods to interact with LLM: invoke/chat model)\n    --structure\n    --templates\n    --specific role assignment\n    --instructions\n    --input examples /Using (RAG)\n    --output format/template\n    --techniques to be used\n    --parameter settings/output generation settings/safety settings\n    --request for reasoning/validation/citations in terms of explainations from LLM for the response it generated)\n> prompt will use analysis + question to be sent > to LLM\n--prompt = PromptTemplate(template = scenario,input_variables=[\'analysis\',\'question\'])\n--create a chain.. LLMChain(prompt=prompt,llm=llm defn)\n--question = \'Look at my analysis and gi

In [32]:
#Setup
#Installation of packages or using a virtual environment with all libraries and packages installed

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('sales_data.csv')

In [4]:
df.describe()

,Sales,Customer_Age,Customer_Satisfaction
count,2500.000000,2500.000000,2500.000000
mean,553.288000,43.332800,3.025869
std,260.101758,14.846758,1.156981
min,100.000000,18.000000,1.005422
25%,324.750000,31.000000,2.056014
50%,552.500000,43.000000,3.049480
75%,779.000000,56.000000,4.042481
max,999.000000,69.000000,4.999006


In [5]:
df.columns

Index(['Date', 'Product', 'Region', 'Sales', 'Customer_Age', 'Customer_Gender',
       'Customer_Satisfaction'],
      dtype='object')

In [6]:
total_sales = df['Sales'].sum()


In [7]:
total_sales

1383220

In [8]:
avg_sale = df['Sales'].mean()
median_sale = df['Sales'].median()
sales_std = df['Sales'].std()

In [9]:
total_sales,avg_sale,median_sale,sales_std

(1383220, 553.288, 552.5, 260.1017582136852)

In [10]:
df.head()

,Date,Product,Region,Sales,Customer_Age,Customer_Gender,Customer_Satisfaction
0,1/1/2022,Widget C,South,786,26,Male,2.874407
1,1/2/2022,Widget D,East,850,29,Male,3.365205
2,1/3/2022,Widget A,North,871,40,Female,4.547364
3,1/4/2022,Widget C,South,464,31,Male,4.555420
4,1/5/2022,Widget C,South,262,50,Female,3.982935


In [11]:
df['Month'] = pd.to_datetime(df['Date']).dt.month

In [12]:
df.head()
#df['Month'].unique()

,Date,Product,Region,Sales,Customer_Age,Customer_Gender,Customer_Satisfaction,Month
0,1/1/2022,Widget C,South,786,26,Male,2.874407,1
1,1/2/2022,Widget D,East,850,29,Male,3.365205,1
2,1/3/2022,Widget A,North,871,40,Female,4.547364,1
3,1/4/2022,Widget C,South,464,31,Male,4.555420,1
4,1/5/2022,Widget C,South,262,50,Female,3.982935,1


In [13]:
monthly_sales = df.groupby('Month', observed=False)['Sales'].sum().sort_values(ascending=False)

In [14]:
monthly_sales

Month
8     124264
5     123646
10    119428
1     118537
7     118493
4     117689
6     115935
3     115367
2     113135
9     111615
11    102775
12    102336
Name: Sales, dtype: int64

In [15]:
best_month = monthly_sales.index[0]

In [16]:
best_month

8

In [17]:
print(best_month)

8


In [18]:
def generate_advanced_data_summary(df):
    # Ensure 'Date' is in datetime format
    df['Date'] = pd.to_datetime(df['Date'])

    # Sales Analysis
    total_sales = df['Sales'].sum()
    avg_sale = df['Sales'].mean()
    median_sale = df['Sales'].median()
    sales_std = df['Sales'].std()

    # Time-based Analysis
    df['Month'] = pd.to_datetime(df['Date']).dt.month
    monthly_sales = df.groupby('Month', observed=False)['Sales'].sum().sort_values(ascending=False)
    best_month = monthly_sales.index[0]
    worst_month = monthly_sales.index[-1]

    # Product Analysis
    product_sales = df.groupby('Product', observed=False)['Sales'].agg(['sum', 'count', 'mean'])
    top_product = product_sales['sum'].idxmax()
    most_sold_product = product_sales['count'].idxmax()

    # Regional Analysis
    region_sales = df.groupby('Region', observed=False)['Sales'].sum().sort_values(ascending=False)
    best_region = region_sales.index[0]
    worst_region = region_sales.index[-1]

    # Customer Analysis
    avg_satisfaction = df['Customer_Satisfaction'].mean()
    satisfaction_std = df['Customer_Satisfaction'].std()

    age_bins = [0, 25, 35, 45, 55, 100]
    age_labels = ['18-25', '26-35', '36-45', '46-55', '55+']
    df['Age_Group'] = pd.cut(df['Customer_Age'], bins=age_bins, labels=age_labels, right=False)
    age_group_sales = df.groupby('Age_Group', observed=False)['Sales'].mean().sort_values(ascending=False)
    best_age_group = age_group_sales.index[0]

    # Gender Analysis
    gender_sales = df.groupby('Customer_Gender', observed=False)['Sales'].mean()

    summary = f"""
    Advanced Sales Data Summary:

    Overall Sales Metrics:
    - Total Sales: ${total_sales:,.2f}
    - Average Sale: ${avg_sale:.2f}
    - Median Sale: ${median_sale:.2f}
    - Sales Standard Deviation: ${sales_std:.2f}

    Time-based Analysis:
    - Best Performing Month: {best_month}
    - Worst Performing Month: {worst_month}

    Product Analysis:
    - Top Selling Product (by value): {top_product}
    - Most Frequently Sold Product: {most_sold_product}

    Regional Performance:
    - Best Performing Region: {best_region}
    - Worst Performing Region: {worst_region}

    Customer Insights:
    - Average Customer Satisfaction: {avg_satisfaction:.2f}/5
    - Customer Satisfaction Standard Deviation: {satisfaction_std:.2f}
    - Best Performing Age Group: {best_age_group}
    - Gender-based Average Sales: Male=${gender_sales['Male']:.2f}, Female=${gender_sales['Female']:.2f}


    Key Observations:
    1. The sales data shows significant variability with a standard deviation of ${sales_std:.2f}.
    2. The {best_age_group} age group shows the highest average sales.
    3. Regional performance varies significantly, with {best_region} outperforming {worst_region}.
    4. The most valuable product ({top_product}) differs from the most frequently sold product ({most_sold_product}), suggesting potential for targeted marketing strategies.
    """

    return summary

In [19]:
print(generate_advanced_data_summary(df))


    Advanced Sales Data Summary:

    Overall Sales Metrics:
    - Total Sales: $1,383,220.00
    - Average Sale: $553.29
    - Median Sale: $552.50
    - Sales Standard Deviation: $260.10

    Time-based Analysis:
    - Best Performing Month: 8
    - Worst Performing Month: 12

    Product Analysis:
    - Top Selling Product (by value): Widget A
    - Most Frequently Sold Product: Widget A

    Regional Performance:
    - Best Performing Region: West
    - Worst Performing Region: East

    Customer Insights:
    - Average Customer Satisfaction: 3.03/5
    - Customer Satisfaction Standard Deviation: 1.16
    - Best Performing Age Group: 18-25
    - Gender-based Average Sales: Male=$547.56, Female=$558.96


    Key Observations:
    1. The sales data shows significant variability with a standard deviation of $260.10.
    2. The 18-25 age group shows the highest average sales.
    3. Regional performance varies significantly, with West outperforming East.
    4. The most valuable produ

In [20]:
summary = generate_advanced_data_summary(df)

In [52]:
#print(summary)

In [53]:
#Generated code from chatGPT
import pandas as pd
import numpy as np

def analyze_dataset_for_llm(df):

    report = {}

    # -----------------------------
    # 1. Dataset Overview
    # -----------------------------
    report["dataset_overview"] = {
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": list(df.columns)
    }

    # -----------------------------
    # 2. Data Types
    # -----------------------------
    report["data_types"] = {
        col: str(dtype) for col, dtype in df.dtypes.items()
    }

    # -----------------------------
    # 3. Missing Values
    # -----------------------------
    report["missing_values"] = {
        col: int(df[col].isna().sum())
        for col in df.columns
    }

    # -----------------------------
    # 4. Numeric Statistics
    # -----------------------------
    numeric_cols = df.select_dtypes(include=np.number).columns

    report["numeric_summary"] = {}

    for col in numeric_cols:
        report["numeric_summary"][col] = {
            "mean": float(df[col].mean()),
            "median": float(df[col].median()),
            "std": float(df[col].std()),
            "min": float(df[col].min()),
            "max": float(df[col].max())
        }

    # -----------------------------
    # 5. Categorical Analysis
    # -----------------------------
    cat_cols = df.select_dtypes(include="object").columns

    report["categorical_summary"] = {}

    for col in cat_cols:

        report["categorical_summary"][col] = {
            "unique_values": int(df[col].nunique()),
            "top_values": df[col].value_counts().head(5).to_dict()
        }

    # -----------------------------
    # 6. Date Analysis
    # -----------------------------
    if "Date" in df.columns:

        df["Date"] = pd.to_datetime(df["Date"])

        report["time_analysis"] = {
            "date_start": str(df["Date"].min()),
            "date_end": str(df["Date"].max()),
            "records_per_year":
                df.groupby(df["Date"].dt.year).size().to_dict()
        }

    # -----------------------------
    # 7. Business Metrics
    # -----------------------------
    if "Sales" in df.columns:

        report["sales_metrics"] = {
            "total_sales": float(df["Sales"].sum()),
            "avg_sales": float(df["Sales"].mean()),
            "median_sales": float(df["Sales"].median())
        }

    if "Product" in df.columns and "Sales" in df.columns:

        report["sales_by_product"] = (
            df.groupby("Product")["Sales"]
            .sum()
            .sort_values(ascending=False)
            .to_dict()
        )

    if "Region" in df.columns and "Sales" in df.columns:

        report["sales_by_region"] = (
            df.groupby("Region")["Sales"]
            .sum()
            .sort_values(ascending=False)
            .to_dict()
        )

    # -----------------------------
    # 8. Customer Insights
    # -----------------------------
    if "Customer_Age" in df.columns:

        report["age_distribution"] = {
            "min": int(df["Customer_Age"].min()),
            "max": int(df["Customer_Age"].max()),
            "avg": float(df["Customer_Age"].mean())
        }

    if "Customer_Gender" in df.columns:

        report["gender_distribution"] = (
            df["Customer_Gender"]
            .value_counts()
            .to_dict()
        )

    if "Customer_Satisfaction" in df.columns:

        report["satisfaction_metrics"] = {
            "avg_satisfaction":
                float(df["Customer_Satisfaction"].mean())
        }

    # -----------------------------
    # 9. Correlation Analysis
    # -----------------------------
    if len(numeric_cols) > 1:

        report["correlations"] = (
            df[numeric_cols]
            .corr()
            .round(3)
            .to_dict()
        )

    return report

In [21]:
"""import pandas as pd
df = pd.read_csv("sales_data.csv")
summary = analyze_dataset_for_llm(df)
#print(summary)"""

'import pandas as pd\ndf = pd.read_csv("sales_data.csv")\nsummary = analyze_dataset_for_llm(df)\n#print(summary)'

In [22]:
import langchain
import langchain_core
import langchain_community
print(langchain.__version__)
print(langchain_core.__version__)
print(langchain_community.__version__)

1.2.3
1.2.7
0.4.1


In [23]:
# Chains
#from langchain.chains import LLMChain, SimpleSequentialChain --for older versions
from langchain_core.runnables import RunnableSequence
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# Models
from langchain_openai import ChatOpenAI, OpenAI, OpenAIEmbeddings

# Documents
from langchain_core.documents import Document

# Prompts
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
    HumanMessagePromptTemplate,
)

# Messages
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    AIMessage,
)

# Vector stores
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores import FAISS

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
    SentenceTransformersTokenTextSplitter,
)

In [24]:
from langchain_openai import AzureChatOpenAI

In [26]:
import os
from dotenv import load_dotenv
#load_dotenv("E:\\Lesson_2_demos\\.env")
load_dotenv()

## Start by creating an instance of the AzureChatOpenAI class.
client = AzureChatOpenAI(
     azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
     api_key=os.getenv("API_KEY"),
     api_version=os.getenv("AZURE_API_VERSION"),
     #deployment_name="gpt-4.1",
     deployment_name= os.getenv("AZURE_DEPLOYMENT_NAME"),
     temperature=0,
 )


In [29]:
template = """Question: {question}
Answer: Let's think step by step and generate answers 
and no metatdata is needed in response 
and no response_metadata.
Provide reasoning and source for the response generated.
"""
prompt = PromptTemplate(template=template, input_variables=["question"])

# Example questions
questions = [
    "Explain the concept of black holes in simple terms.",
    "What are the main causes of climate change, and how can we address them?",
    "Provide a brief overview of the history of artificial intelligence."
]

In [ ]:
# # RunnableSequence
chain = prompt | client

for q in questions:
     print(f"\nQ: {q}")
     print(chain.invoke({"question": q}))

In [32]:
#summary

In [33]:
#To use our analyzed data
scenario_template = """
You are an AI sales analyst. Use the summary data which we have analyzed, provide some in-depth analysis and 
some recommendation based on data provided. 
Be specific to the data points and give me citations if the recommendations were based on some web or public resources.
    {summary}
Question: {question}

Detaled Analysis & recommendation by AI:
"""

In [34]:
prompt = PromptTemplate(template=scenario_template,input_variables=["summary","question"])

In [35]:
#print(summary)

In [36]:
#newer version
chain = prompt | client
question = "based on this data, what are our areas of improvement"
#To ask user to input data
#question = str(input("enter your question"))
def gen_insight(summary, question):
    return chain.invoke({"summary": summary, "question": question}).content

In [67]:
insight = gen_insight(summary,question)
print(insight)

Certainly! Here’s an in-depth analysis of your sales data, highlighting key areas for improvement, supported by specific data points and actionable recommendations.

---

## **Detailed Analysis**

### 1. **Sales Distribution & Product Performance**
- **Widget A** leads in sales (375,235), followed by Widget B (346,062), Widget C (335,069), and Widget D (326,854).
- The difference between the top and bottom product is **48,381** units, indicating relatively balanced but improvable product performance.

### 2. **Regional Performance**
- **West** region is the top performer (361,383), while **East** lags behind (320,296).
- The gap between West and East is **41,087**, suggesting potential for growth in the East.

### 3. **Customer Demographics**
- **Age**: Average customer age is **43.3** (range: 18–69), indicating a mature customer base.
- **Gender**: Sales are almost evenly split (**Female: 1,256; Male: 1,244**), showing no gender bias in reach.

### 4. **Customer Satisfaction**
- **Ave

In [38]:
#optional
#Splitting tasks in different chains
#Note** Modify code as per langchain version and to use newer method to invoke chain
#from langchain import SequentialChain
##implementation of seq chain to do the same thing as above in 2 different chains followed
#sequentially

#1st
#data_analysis_template = """analyze my {summary} data  and provide a addiitonal detailed summary """
#a_prompt = PromptTemplate(template=data_analysis_template,input_variables=["summary"])
#chain1 = LLMChain(prompt=a_prompt,llm=myllm,output='output')--older version
#chain1 = a_prompt | client (newer version example)

#2nd
#recommendation_template = """based on {analysis} data  and provide a recommendaton to address the question: {question} """
#b_prompt = PromptTemplate(template=recommendation_template,input_variables=["analysis","question"])
#chain2 = LLMChain(prompt=b_prompt,llm=myllm,output='output2')
#chain2 = b_prompt | client (newer version example)

'''overall_chain = SequentialChain(
       chains = [chain1,chain2],
       input_variables=['summary','question'],
       output_variables = ['analysis','recommendation'],
)'''

"overall_chain = SequentialChain(\n       chains = [chain1,chain2],\n       input_variables=['summary','question'],\n       output_variables = ['analysis','recommendation'],\n)"

In [71]:
'''def gen_insight(question):
    result = overall_chain.run({"summary": summary,"question": question})
    return f"Analysis:\n{result['analysis']\n\nRecommendation:\n{result['recommendation']}"'''

'def gen_insight(question):\n    result = overall_chain.run({"summary": summary,"question": question})\n    return f"Analysis:\n{result[\'analysis\']\n\nRecommendation:\n{result[\'recommendation\']}"'

In [72]:
'''question = "how can we improve in a region"'''
'''questions = ['how can we improve in a region','what would be better approach to cater to people from
                a specific region']
   for i in questions:
       print(gen_insight(i))'''
'''insight = gen_insight(question)
print(insight)'''

'insight = gen_insight(question)\nprint(insight)'

In [73]:
#!pip install pypdf

In [39]:
import pypdf

In [40]:
#If not done earlier
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
    SentenceTransformersTokenTextSplitter,
)

In [41]:
import os
os.listdir('PDF Folder')

['.ipynb_checkpoints',
 'AI business model innovation.pdf',
 'BI approaches.pdf',
 'project.ipynb',
 'Time-Series-Data-Prediction-using-IoT-and-Machine-Le_2020_Procedia-Computer-.pdf',
 'Walmarts sales data analysis.pdf']

In [42]:
pdf_folder = 'PDF Folder'

In [43]:
from pypdf import PdfReader

In [44]:
documents = []
for file in os.listdir(pdf_folder):
    if file.endswith('.pdf'):
        print(file)

AI business model innovation.pdf
BI approaches.pdf
Time-Series-Data-Prediction-using-IoT-and-Machine-Le_2020_Procedia-Computer-.pdf
Walmarts sales data analysis.pdf


In [45]:
from pypdf import PdfReader
from langchain_core.documents import Document
import os

documents = []

for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        file_path = os.path.join(pdf_folder, file)

        with open(file_path, "rb") as pdf_file:
            reader = PdfReader(pdf_file)

            for page in reader.pages:
                text = page.extract_text()

                if text:
                    documents.append(
                        Document(
                            page_content=text,
                            metadata={"source": file}
                        )
                    )

In [46]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap= 200)

In [47]:
texts = text_splitter.split_documents(documents)

In [49]:
#texts

In [75]:
#optional
#save your texts as .pkl

In [50]:
from langchain_community.embeddings import HuggingFaceEmbeddings
model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model)

C:\Users\Ajay\AppData\Local\Temp\ipykernel_2016\3834499437.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model)


In [84]:
#If not done already, uncomment and run
#!pip install faiss-cpu

In [51]:
from langchain_community.vectorstores.faiss import FAISS

In [86]:
#Since using newer version we will use RunnablePassthrough instead of 
#--from langchain.chains import RetrievalQA
#---->from langchain.chains import create_retrieval_chain
#      from langchain.chains.combine_documents import create_stuff_documents_chain

In [52]:
vectorstore = FAISS.from_documents(texts, embeddings)

In [55]:
#Testing

all_docs = list(vectorstore.docstore._dict.values())

#uncomment to see results of data from vectorstore
'''for i, doc in enumerate(all_docs):
    print(f"Document {i+1}:\n{doc.page_content}\n")'''

'for i, doc in enumerate(all_docs):\n    print(f"Document {i+1}:\n{doc.page_content}\n")'

In [99]:
retriever = vectorstore.as_retriever(search_type="similarity", k=4)

In [101]:
#in older versions we were using RetrievalQA
'''
qa_chain = RetrievalQA.from_chain_type(
    llm=client,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)'''

'\nqa_chain = RetrievalQA.from_chain_type(\n    llm=client,\n    chain_type="stuff",\n    retriever=retriever,\n    return_source_documents=True\n)'

In [113]:
#in Newer version
#if not done earlier
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [107]:
prompt = ChatPromptTemplate.from_template(
"""Answer the question based only on the context below.

Context:
{context}

Question:
{question}
"""
)

In [108]:
#Retriever returns Document objects, so we convert them to text.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [109]:
#Build the Runnable RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | client
    | StrOutputParser()
)

In [111]:
response = rag_chain.invoke("What is AI business model innovation?")
print(response)

Based only on the context provided:

**AI business model innovation (AI-driven BMI)** refers to the process of innovating or transforming business models through the implementation and integration of artificial intelligence (AI) technologies. The literature highlights that AI-driven BMI has a profound impact across industries, but research in this area is currently fragmented, often focusing mainly on the technological aspects of AI within business models and treating innovation as a secondary effect.

AI-driven BMI involves both static and dynamic perspectives on how business models evolve due to AI, and includes the development of new value propositions, capabilities, and frameworks enabled by AI. It encompasses the ways AI can scale and transform business models through co-evolutionary processes and feedback loops, ultimately leading to new or improved ways of creating, delivering, and capturing value in organizations.


In [112]:
#see retieved documents
docs = retriever.invoke("AI business model innovation")
print(docs[0].page_content)

ARTICLE INFO  
Keywords: 
Business model innovation 
Artificial intelligence 
Value proposition 
AI-driven BMI 
Systematic literature review 
ABSTRACT  
Recent years have seen a surge in research on artificial intelligence (AI)-driven business model innovation (BMI), 
reflecting its profound impact across industries. However, the field ’ s current state remains fragmented due to 
varied conceptual lenses and units of analysis. Existing literature predominantly emphasizes the technological 
aspects of AI implementation in business models (BMs), treating BMI as a byproduct. Additionally, there is a lack 
of coherent understanding regarding the scope of BMI propelled by AI. To address these gaps, our study sys -
tematically reviews 180 articles, offering two key contributions: (1) a structured analysis of evolving research 
dimensions in AI-driven BMI, differentiating between static and dynamic views of BMI, and (2) a framework


In [116]:
#Combining response and retriever
prompt = ChatPromptTemplate.from_template(
"""Answer the question using only the context below.

Context:
{context}

Question:
{question}
"""
)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

answer_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | client
    | StrOutputParser()
)

rag_chain = RunnableParallel(
    answer=answer_chain,
    sources=retriever
)



In [117]:
result = rag_chain.invoke("What is AI business model innovation?")
print("Answer:\n", result["answer"])

print("\nSources:\n")
for doc in result["sources"]:
    print(doc.metadata)

Answer:
 AI business model innovation (AI-driven BMI) refers to the transformation and development of business models propelled by artificial intelligence (AI). It involves leveraging AI capabilities to create new value propositions, redesign organizational processes, and enable novel ways of delivering products and services. While existing literature often focuses on the technological aspects of AI implementation, AI-driven BMI emphasizes how AI can fundamentally reshape business models, either as a byproduct of technology adoption or through intentional innovation. This includes both static and dynamic views of business model change, and encompasses a taxonomy of AI applications and innovation capabilities that allow organizations to scale and adapt through co-evolutionary processes and feedback loops.

Sources:

{'source': 'AI business model innovation.pdf'}
{'source': 'AI business model innovation.pdf'}
{'source': 'AI business model innovation.pdf'}
{'source': 'AI business model in

In [118]:
#Show Source Text 
for doc in result["sources"]:
    print("\n--- Source ---")
    print(doc.metadata["source"])
    print(doc.page_content[:300])


--- Source ---
AI business model innovation.pdf
ARTICLE INFO  
Keywords: 
Business model innovation 
Artificial intelligence 
Value proposition 
AI-driven BMI 
Systematic literature review 
ABSTRACT  
Recent years have seen a surge in research on artificial intelligence (AI)-driven business model innovation (BMI), 
reflecting its profound impact 

--- Source ---
AI business model innovation.pdf
techfore.2023.122903 
Sj ¨odin, D., Parida, V., Palmi ´e, M., & Wincent, J. (2021). How AI capabilities enable 
business model innovation: Scaling AI through co-evolutionary processes and 
feedback loops. Journal of Business Research, 134 (13), 574 – 587. https://doi.org/ 
10.1016/j.jbusres.2021.05.

--- Source ---
AI business model innovation.pdf
review of innovation capabilities and a taxonomy of AI applications. Journal of 
Product Innovation Management, Article jpim.12698. Advance online publication. . 
https://doi.org/10.1111/jpim.12698 
Garbuio, M., & Lin, N. (2019). Artificial Intelligen

In [119]:
#define a function to use wiki search or ay other tool to handle dynamic questions

In [125]:
question = "what are top selling products"
result = rag_chain.invoke(question)
print(result["answer"])

The provided context does not specify the exact top selling products at Walmart. It discusses the use of big data analytics to analyze sales data, forecast trends, and understand business drivers, but does not list specific products.


In [130]:
#changing context not to be from retiever and to accept context directly
#Combining response and retriever
prompt = ChatPromptTemplate.from_template(
"""Answer the question using only the context below.

Context:
{context}

Question:
{question}
"""
)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

answer_chain = (
    prompt
    | client
    | StrOutputParser()
)

#Keep the retriever only for sources
rag_chain = RunnableParallel(
    answer=answer_chain,
    sources=lambda x: retriever.invoke(x["question"])
)


In [131]:
#building context from summary
question = "what are top selling products"

context = f"""
My summary data is:

{summary}
"""

In [132]:
result = rag_chain.invoke({
    "context": context,
    "question": question
})


In [133]:
result

{'answer': 'The top selling products are:\n\n1. Widget A (375,235 in sales)\n2. Widget B (346,062 in sales)\n3. Widget C (335,069 in sales)\n4. Widget D (326,854 in sales)',
 'sources': [Document(id='1fe7ea75-5171-4e71-b48f-3a69722041f8', metadata={'source': 'Walmarts sales data analysis.pdf'}, page_content='Walmart’s Sales Data Analysis- A Big Data\nAnalytics Perspective\nManpreet Singh∗§, Bhawick Ghutla †, Reuben Lilo Jnr †, Aesaan F S Mohammed † and Mahmood A Rashid †‡§\n∗National Training and Productivity Centre, Fiji National University, Samabula, Suva, Fiji\n†School of Computing, Information and Mathematical Sciences, The University of the South Paciﬁc, Suva, Fiji\n‡Institute for Integrated and Intelligent Systems, Grifﬁth University, QLD, Australia\n§Corresponding Authors: manpreet.singh@fnu.ac.fj OR mahmood.rashid@usp.ac.fj\nAbstract—Information technology in this 21st century is reaching\nthe skies with large-scale of data to be processed and studied\nto make sense of data whe

In [ ]:
#Eval Chain

In [ ]:
#QAEvalChain
#from langchain.evaluation.qa import QAEvalChain
eval_chain = QAEvalChain.from_llm(client)

examples = [
        {
        "query":  "what is the growth in my sales",
            "Answer" : "sales has been stagnant over years"
        }
]

prediction = [
    {
        "response": "response generated..."
    }
]

results = eval_chain.evaluate(examples, predictions)

#==========================
Question: {question}
Ground truth: {reference}
LLM response: {prediction}
examples: ...

#Correctness, failthfulness, completeness



In [ ]:
#visualization